# RAG: выполненный эксперимент

Выполнено 25.09.2026 в Colab на T4: три режима по 50 вопросов. Выводы сохранены. Это журнал исходного прогона с исправлениями зависимостей; для запуска с нуля используйте 02_reproduce.ipynb.


In [1]:
from pathlib import Path
import os, sys, json, subprocess, zipfile
root = Path('/content/rag-homework')
with zipfile.ZipFile('/content/rag-homework-data.zip') as archive:
    archive.extractall('/content')
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
subprocess.run([sys.executable, '-m', 'src.run', '--mode', 'validate'], check=True)
from google.colab import userdata
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
import requests
response = requests.get('https://api.groq.com/openai/v1/models', headers={'Authorization': 'Bearer ' + os.environ['GROQ_API_KEY']}, timeout=30)
print('Groq status:', response.status_code)
if response.ok:
    print('Available models:', sorted(x['id'] for x in response.json()['data']))
else:
    print('Groq access failed; key is not displayed.')
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], check=True)


Groq status: 200
Available models: ['allam-2-7b', 'canopylabs/orpheus-arabic-saudi', 'canopylabs/orpheus-v1-english', 'meta-llama/llama-prompt-guard-2-22m', 'meta-llama/llama-prompt-guard-2-86m', 'openai/gpt-oss-120b', 'openai/gpt-oss-20b', 'openai/gpt-oss-safeguard-20b', 'qwen/qwen3.8-27b', 'whisper-large-v3', 'whisper-large-v3-turbo']


CompletedProcess(args=['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], returncode=0)

In [2]:
config_path = Path('configs/default.json')
config = json.loads(config_path.read_text())
config.update(groq_model='openai/gpt-oss-20b', judge_model='openai/gpt-oss-120b', max_tokens=2048, local_max_tokens=256)
config_path.write_text(json.dumps(config, ensure_ascii=False, indent=2))
p = Path('src/models.py')
s = p.read_text()
s = s.replace('temperature=config["temperature"], max_tokens=512 if judge else config["max_tokens"],', 'model_kwargs={"reasoning_effort": "low"}, temperature=config["temperature"], max_tokens=config["max_tokens"],')
s = s.replace('gpu_memory_utilization=0.8, seed=', 'gpu_memory_utilization=0.8, enforce_eager=True, seed=')
s = s.replace('max_tokens=config["max_tokens"], seed=', 'max_tokens=config["local_max_tokens"], seed=')
p.write_text(s)
install = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt', 'vllm==0.11.0'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(install.stdout[-8000:])
assert install.returncode == 0, 'Dependency installation failed'
print('Dependencies installed')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.2/438.2 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.0/180.0 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 94.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 93.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.

## Данные и ключ
Файлы лежат в data/. В Colab добавьте GROQ_API_KEY в Secrets и разрешите этому ноутбуку доступ.

In [3]:
from google.colab import userdata
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
config = json.loads(Path('configs/default.json').read_text())
from src.data import load_dataset
articles, questions, references = load_dataset(config)
print(f'Validated: {len(articles)} articles, {len(questions)} questions, {len(references)} references')
def run_stage(mode):
    command = [sys.executable, '-u', '-m', 'src.run', '--mode', mode]
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
    assert code == 0, f'{mode} failed; saved checkpoints can be resumed'
print('Runner ready')


Validated: 10 articles, 50 questions, 50 references
Runner ready


In [4]:
print(json.dumps(config, ensure_ascii=False, indent=2))


{
  "data_dir": "data",
  "results_dir": "results",
  "expected_articles": 10,
  "expected_questions": 50,
  "groq_model": "openai/gpt-oss-20b",
  "judge_model": "openai/gpt-oss-120b",
  "local_model": "Qwen/Qwen2.5-1.5B-Instruct",
  "embedding_model": "intfloat/multilingual-e5-small",
  "chunk_size": 700,
  "chunk_overlap": 120,
  "top_k": 4,
  "max_tokens": 2048,
  "temperature": 0,
  "seed": 42,
  "request_interval": 2,
  "local_max_tokens": 256
}


## API-модель без RAG

In [6]:
p = Path('src/models.py')
s = p.read_text().replace('model_kwargs={"reasoning_effort": "low"}', 'reasoning_effort="low"')
p.write_text(s)
run_stage('groq_zero')


groq_zero: 1/50
groq_zero: 2/50
groq_zero: 3/50
groq_zero: 4/50
groq_zero: 5/50
groq_zero: 6/50
groq_zero: 7/50
groq_zero: 8/50
groq_zero: 9/50
groq_zero: 10/50
groq_zero: 11/50
groq_zero: 12/50
groq_zero: 13/50
groq_zero: 14/50
groq_zero: 15/50
groq_zero: 16/50
groq_zero: 17/50
groq_zero: 18/50
groq_zero: 19/50
groq_zero: 20/50
groq_zero: 21/50
groq_zero: 22/50
groq_zero: 23/50
groq_zero: 24/50
groq_zero: 25/50
groq_zero: 26/50
groq_zero: 27/50
groq_zero: 28/50
groq_zero: 29/50
groq_zero: 30/50
groq_zero: 31/50
groq_zero: 32/50
groq_zero: 33/50
groq_zero: 34/50
groq_zero: 35/50
groq_zero: 36/50
groq_zero: 37/50
groq_zero: 38/50
groq_zero: 39/50
groq_zero: 40/50
groq_zero: 41/50
groq_zero: 42/50
groq_zero: 43/50
groq_zero: 44/50
groq_zero: 45/50
groq_zero: 46/50
groq_zero: 47/50
groq_zero: 48/50
groq_zero: 49/50
groq_zero: 50/50
{
  "rouge1": 0.22868992872829427,
  "rouge2": 0.11249615323639336,
  "rougeL": 0.209897627278219,
  "bleu": 3.547397910829684,
  "bleu_signature": "nrefs:1|ca

## Локальная модель через vLLM
Отдельный процесс освобождает GPU после завершения.

In [8]:
fix = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.57.1'], capture_output=True, text=True)
print(fix.stdout[-2000:] + fix.stderr[-2000:])
assert fix.returncode == 0
run_stage('local_zero')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 110.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 102.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.

INFO 09-25 11:12:19 [__init__.py:216] Automatically detected platform cuda.
2026-09-25 11:12:22.353178: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the f

## Та же API-модель с RAG

In [9]:
run_stage('groq_rag')


2026-09-25 11:20:27.553644: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Index: 10 articles, 30 chunks
groq_rag: 1/50
groq_rag: 2/50
groq_rag: 3/50
groq_rag: 4/50
groq_rag: 5/50
groq_rag: 6/50
groq_rag: 7/50
groq_rag: 8/50
groq_rag: 9/50
groq_rag: 10/50
groq_rag: 11/50
groq_rag: 12/50
groq_rag: 13/50
groq_rag: 14/50
groq_rag: 15/50
groq_rag: 16/50
groq_rag: 17/50
groq_rag: 18/50
groq_rag: 19/50
groq_rag: 20/50
groq_rag: 21/50
groq_rag: 22/50
groq_rag: 23/50
groq_rag: 24/50
groq_rag: 25/50
groq_rag: 26/50
groq_rag: 27/50
groq_rag: 28/50
groq_rag: 29/50
groq_rag: 30/50
groq_rag: 31/50
groq_rag: 32/50
groq_rag: 33/50
groq_rag: 34/50
groq_rag: 35/50
groq_rag: 36/50
groq_rag: 37/50
groq_rag: 38/50
groq_rag: 39/50
groq_rag: 40/50
groq_rag: 41/50
gro

## Таблица, ошибки и выводы

In [10]:
report = subprocess.run([sys.executable, '-m', 'src.report'], capture_output=True, text=True)
assert report.returncode == 0, report.stderr
from IPython.display import Markdown, display
display(Markdown(Path(config['results_dir'], 'REPORT.md').read_text(encoding='utf-8')))


# Результаты эксперимента

| Режим | N | BLEU /100 | ROUGE-1 | ROUGE-2 | ROUGE-L | Judge /4 |
|---|---:|---:|---:|---:|---:|---:|
| groq_zero | 50 | 3.55 | 0.229 | 0.112 | 0.210 | 0.64 |
| local_zero | 50 | 3.31 | 0.225 | 0.100 | 0.208 | 0.24 |
| groq_rag | 50 | 36.60 | 0.659 | 0.503 | 0.622 | 3.68 |

При добавлении RAG средняя оценка судьи изменилась на +3.04 балла из 4.
Сравнение использует одну и ту же API-модель и одинаковые вопросы. Это наблюдение на данном наборе, не доказательство статистической значимости.

## Примеры ошибок RAG

### Вопрос q1
За сколько рабочих дней до поездки нужно подать заявку на командировку в TravelHub?

Ответ: В базе знаний нет ответа.

Эталон: Заявку нужно подать не позднее чем за 5 рабочих дней до выезда.

Судья: Ответ не содержит информации, в отличие от референса, где указано 5 рабочих дней

### Вопрос q12
Была ли выявлена утечка данных клиентов по итогам расследования?

Ответ: В базе знаний нет ответа.

Эталон: Нет, утечка данных клиентов не выявлена.

Судья: Ответ не содержит информации о наличии утечки, в отличие от референса, который утверждает отсутствие утечки.

### Вопрос q19
Какой целевой месячный процент соблюдения SLA установлен в политике?

Ответ: В базе знаний нет ответа.

Эталон: Целевой месячный процент соблюдения SLA — 97,5%.

Судья: Ответ не содержит нужной информации, в отличие от референса, где указан целевой процент 97,5%.

### Вопрос q41
Какой риск по росту расходов прогнозируется во втором полугодии?

Ответ: В базе знаний нет ответа.

Эталон: Прогнозируется рост расходов до 18% во втором полугодии без допмер.

Судья: Ответ не содержит информации, противоречит референсу, который указывает рост расходов до 18%

### Вопрос q2
Какой лимит на проживание по России установлен в регламенте командировок?

Ответ: Лимит на проживание по России установлен в размере **9000 рублей за ночь**.

Эталон: Лимит на проживание по России составляет 9000 рублей за ночь.

Судья: Ответ полностью совпадает с референсом, точный и полный

## Ограничения и улучшения

BLEU и ROUGE измеряют совпадения слов и не учитывают все допустимые перефразировки. LLM-судья может ошибаться; нужна ручная проверка ошибок и второй независимый судья. Далее: отдельный validation set для выбора chunk_size/top_k, гибридный BM25+dense поиск, reranker, разметка релевантных документов для Recall@k и проверка обоснованности ответа источниками.


## Сохранение
В Colab: Файл → Скачать → .ipynb. Проверьте, что таблица и ответы видны после повторного открытия. Сохраните results вместе с ноутбуком. Не публикуйте ключи.

## Примечание к исходному отчёту
В исходном выводе выше показаны пять ответов с наименьшими оценками. Ошибок только четыре: q1, q12, q19, q41; q2 корректен. Финальный results/REPORT.md и src/report.py исправлены: включают только оценки ниже 4. См. results/ERROR_ANALYSIS.md. Реальные ответы и метрики не изменялись.
